In [ ]:
# Colab cell: подготовка
!git clone https://github.com/your/repo.git lama_repo  # или клонируй свой форк
%cd lama_repo

# Установки (минимум)
!pip install -q xarray netcdf4 hydra-core omegaconf torch torchvision

# Добавляем путь
import sys

sys.path.append("/content/lama_repo")  # скорректируй путь

# Programmatic Hydra: compose + instantiate dataset, затем синхронизировать in_channels и создать модель
from hydra import initialize, compose
import hydra.utils
from omegaconf import OmegaConf

with initialize(config_path="configs", job_name="colab_job"):
    cfg = compose(config_name="training/lama_small_netcdf.yaml")

# Переопределим путь к netCDF и список переменных (можно сделать overrides в compose, но здесь явно)
cfg.data.dataset.nc_path = "/content/data/BALTICSEA_ANALYSISFORECAST_PHY_003_006.nc"
cfg.data.dataset.var_names = ["temperature", "salinity"]

# Инстанцируем датасет через hydra (использует _target_ в data config)
dataset = hydra.utils.instantiate(cfg.data.dataset, settings=cfg.settings)

# Проверка формата
sample = dataset[0]
print("sample image shape:", sample["image"].s  hape)  # C,H,W
print("sample mask shape:", sample["mask"].shape)

# Синхронизируем модельный конфиг: выставим in_channels = len(var_names)
in_ch = len(cfg.data.dataset.var_names)
# cfg.model может быть ссылкой; получим модельный конфиг и установим поле
# Если cfg.model._target_ присутствует, просто изменим generator.in_channels
if "generator" in cfg.model:
    cfg.model.generator.in_channels = in_ch
else:
    cfg.model.in_channels = in_ch

# Инстанцируем модель через hydra (cfg.model должен содержать _target_ или ссылку на фабрику)
# В нашем примере model._target_ = models.factory.build_model
model = hydra.utils.instantiate(cfg.model)

# Быстрый тестовый шаг обучения (1 батч)
from torch.utils.data import DataLoader
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

loader = DataLoader(
    dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=cfg.training.num_workers
)
optim = torch.optim.Adam(model.parameters(), lr=cfg.training.optimizer.lr)

model.to(device)
model.train()
for batch in loader:
    img = batch["image"].to(device)  # shape: B,C,H,W expected
    mask = batch["mask"].to(device)
    # Если dataset возвращает C,H,W, DataLoader добавит batch dim
    out = model(img, mask)  # интерфейс зависит от реализации модели
    loss = ((out - img) ** 2).mean()
    optim.zero_grad()
    loss.backward()
    optim.step()
    print("loss:", loss.item())
    break
